# Graph Neural Network (GNN) - PyTorch

**Goal:** Classify nodes in a toy citation-style graph.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** Nodes update by aggregating transformed neighbor features.
- **Where it is used:** social networks, molecules, recommendations, and citation graphs.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Graph Neural Network: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['nodes', 'neighbors', 'embeds']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = np.cos(x)*np.exp(-.1*x**2)
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.25,.25,.30,.20])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['n0', 'n1', 'n2', 'n3'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
nodes, features, classes = 120, 16, 3
X = torch.randn(nodes, features, device=device)
labels = ((X[:, 0] + X[:, 1] * 0.5) > 0).long() + (X[:, 2] > 1.0).long()
labels = labels.clamp(max=classes - 1)

adjacency = torch.eye(nodes, device=device)
for i in range(nodes):
    neighbors = torch.randperm(nodes, device=device)[:5]
    adjacency[i, neighbors] = 1
    adjacency[neighbors, i] = 1
degree_inv_sqrt = adjacency.sum(1).pow(-0.5)
norm_adj = degree_inv_sqrt[:, None] * adjacency * degree_inv_sqrt[None, :]
train_idx = torch.arange(0, 80, device=device)
test_idx = torch.arange(80, nodes, device=device)


In [ ]:
class GraphConvolution(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x, normalized_adjacency):
        return normalized_adjacency @ self.linear(x)


class GCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.gcn1 = GraphConvolution(features, 32)
        self.gcn2 = GraphConvolution(32, classes)

    def forward(self, x, normalized_adjacency):
        x = torch.relu(self.gcn1(x, normalized_adjacency))
        return self.gcn2(x, normalized_adjacency)


model = GCN().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2, weight_decay=5e-4)


In [ ]:
for epoch in range(120):
    logits = model(X, norm_adj)
    loss = nn.functional.cross_entropy(logits[train_idx], labels[train_idx])
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 30 == 0:
        acc = (logits[test_idx].argmax(1) == labels[test_idx]).float().mean().item()
        print(f"epoch={epoch+1:03d} test_accuracy={acc:.3f}")
